# RAG Evaluation: With SVM Classifier vs Without

Evaluates the RAG pipeline in two modes:
- **With ML**: SVM + TF-IDF → category filter on Pinecone + per-category BM25 → Cohere rerank → Groq LLM
- **Without ML**: All-category Pinecone + BM25 → Cohere rerank → Groq LLM (no classifier)

Metrics: retrieval type precision, type diversity, embedding similarity vs ground truth.

Test set: 8 queries (2 per type: Incident, Request, Problem, Change).

In [1]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="minimax-m3:cloud")

res = llm.invoke("hello")
res

AIMessage(content='Hello! 👋 How can I help you today?', additional_kwargs={}, response_metadata={'model': 'minimax-m3', 'created_at': '2026-07-11T08:22:34.357271037Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1551981775, 'load_duration': None, 'prompt_eval_count': 177, 'prompt_eval_duration': None, 'eval_count': 77, 'eval_duration': None, 'logprobs': None, 'model_name': 'minimax-m3', 'model_provider': 'ollama'}, id='lc_run--019f5045-9313-7ef0-911e-b9ab0cf2db11-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 177, 'output_tokens': 77, 'total_tokens': 254})

In [2]:
import asyncio
import json
import sys
import time
import numpy as np
import pandas as pd
from pathlib import Path
from random import seed, shuffle

BACKEND_DIR = Path('.').resolve()
while not (BACKEND_DIR / 'app').exists() and BACKEND_DIR.parent != BACKEND_DIR:
    BACKEND_DIR = BACKEND_DIR.parent
sys.path.insert(0, str(BACKEND_DIR))

from app.classifier.model import Classifier
from app.rag.retriever import Retriever
from app.rag.generator import Generator
from app.config import settings

TYPES = ['Incident', 'Request', 'Problem', 'Change']

c:\Files\Customer Support Ticket System\backend\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
FALLBACK_RESULTS_JSON = '''[
  {"query": "System Integration for Software Update...", "expected_type": "Change", "predicted_type": "Change", "category_correct": true, "ml": {"answer": "**Answer:** \\nTo proceed with the system update integration for your financial firm, we require more information about your current system software. This will enable us to provide you with tailored guidance on the necessary steps and timelines for the integration process.\\n\\n**Steps:**\\n\\n1. Please share", "retrieval": {"type_precision": 1.0, "type_diversity": 1}, "similarity": 0.791}, "no_ml": {"answer": "**Answer:** \\nTo assist you with the system update integration for your financial firm, we would like to provide you with information about the process, including the required steps and timelines. However, to better support your request, could you please share details about your current system softwa", "retrieval": {"type_precision": 1.0, "type_diversity": 1}, "similarity": 0.914}},
  {"query": "Securing Medical Data in Salesforce CRM with Django 3.2...", "expected_type": "Request", "predicted_type": "Request", "category_correct": true, "ml": {"answer": "**Answer:** To ensure the confidentiality, integrity, and availability of sensitive medical information in the Salesforce CRM Django 3.2 integration, we recommend implementing encryption, access controls, and auditing. For the integration of Django 3.2 with Salesforce CRM, consider using secure APIs", "retrieval": {"type_precision": 1.0, "type_diversity": 1}, "similarity": 0.908}, "no_ml": {"answer": "**Answer:**\\nTo ensure the confidentiality, integrity, and availability of sensitive medical information in the Salesforce CRM integrated with Django 3.2, we recommend implementing encryption, access controls, and auditing. For the Django 3.2 Salesforce CRM integration, consider using secure APIs and", "retrieval": {"type_precision": 1.0, "type_diversity": 1}, "similarity": 0.868}},
  {"query": "Terraform Integration in Data Analytics Assistance...", "expected_type": "Request", "predicted_type": "Request", "category_correct": true, "ml": {"answer": "**Answer:**\\nYes, to get details on the Terraform integration for data analytics investment optimization, you can visit our website for detailed documentation on the integration process. This includes setting up the Terraform plugin for KNIME and reviewing the configuration guide.\\n\\nSteps:\\n\\n1. Visit our w", "retrieval": {"type_precision": 1.0, "type_diversity": 1}, "similarity": 0.771}, "no_ml": {"answer": "**Answer:**\\nTo get started with Terraform integration for data analytics investment optimization, you can visit our website for detailed documentation on the integration process. This involves setting up the Terraform plugin for KNIME and reviewing the configuration guide.\\n\\nSteps:\\n\\n1. Visit our website", "retrieval": {"type_precision": 1.0, "type_diversity": 1}, "similarity": 0.758}},
  {"query": "Technische Sprengung...", "expected_type": "Incident", "predicted_type": "Problem", "category_correct": false, "ml": {"answer": "**Answer:**\\nUm den kritischen Fehler in der Projektmanagement-Software zu beheben, ben\u00f6tigen wir weitere Informationen zu den Fehlermeldungen und den genauen Zeiten des Ausfalls. Bitte teilen Sie uns die Details der Fehlermeldungen und die genauen Zeiten des Ausfalls mit, damit wir die Untersuchung eff", "retrieval": {"type_precision": 0.0, "type_diversity": 1}, "similarity": 0.575}, "no_ml": {"answer": "**Answer:**\\nUm das technische Problem mit der Projektmanagement-Software zu l\u00f6sen, ben\u00f6tigen wir weitere Informationen zu den Fehlermeldungen und der Systemkonfiguration. Bitte geben Sie uns genaue Details zu den Fehlermeldungen, die angezeigt werden, sowie Informationen \u00fcber die Systemkonfiguration, e", "retrieval": {"type_precision": 1.0, "type_diversity": 1}, "similarity": 0.583}},
  {"query": "Problem with Network and Digital Tools...", "expected_type": "Problem", "predicted_type": "Problem", "category_correct": true, "ml": {"answer": "**Answer:**\\nTo address the connectivity problems with Mesh-Netzwerk and digital tools, please contact us at <tel_num> for urgent assistance. We require your account number, <acc_num>, to efficiently troubleshoot and resolve the network issue.\\n\\nSteps:\\n\\n1. Call <tel_num> to report the issue.\\n2. Provide yo", "retrieval": {"type_precision": 1.0, "type_diversity": 1}, "similarity": 0.925}, "no_ml": {"answer": "**Answer:** To address the network issue affecting Mesh-Netzwerk and digital tools access, please contact us at <tel_num> for troubleshooting. We require details on the affected tools and software to assist effectively.\\n\\n**Steps:**\\n\\n1. Prepare a list of the affected digital tools and software.\\n2. Co", "retrieval": {"type_precision": 0.4, "type_diversity": 2}, "similarity": 0.881}},
  {"query": "Enhancements for Data Analysis Tools...", "expected_type": "Change", "predicted_type": "Change", "category_correct": true, "ml": {"answer": "**Answer:** \\nThank you for reaching out to our support team regarding enhancements to the data analytics tools. We appreciate your feedback and are interested in learning more about the specific features you would like to see, such as predictive data analysis and machine learning capabilities. \\n\\n**S", "retrieval": {"type_precision": 1.0, "type_diversity": 1}, "similarity": 0.750}, "no_ml": {"answer": "**Answer:** \\nThank you for reaching out to our support team regarding enhancements to the data analytics tools. We appreciate your feedback and are interested in learning more about the specific features you would like to see, such as predictive data analysis and machine learning capabilities. \\n\\n**S", "retrieval": {"type_precision": 1.0, "type_diversity": 1}, "similarity": 0.770}},
  {"query": "Urgent Security Breach Alert...", "expected_type": "Incident", "predicted_type": "Incident", "category_correct": true, "ml": {"answer": "**Answer:** \\nWe acknowledge your report of an urgent security breach affecting various devices, including projectors, screens, storage units, and cloud services. We understand the critical nature of this situation and are prioritizing your case.\\n\\n**Steps:**\\n\\n1. Please provide additional details such", "retrieval": {"type_precision": 1.0, "type_diversity": 1}, "similarity": 0.913}, "no_ml": {"answer": "**Answer:** \\nTo support our immediate investigation into the urgent security breach, please provide additional details such as specific logs from the impacted devices (projectors, screens, storage units, and cloud platforms), including timestamps of suspicious activities, error messages, or unusual ", "retrieval": {"type_precision": 1.0, "type_diversity": 1}, "similarity": 0.924}},
  {"query": "Customer Inquiry Regarding Unexpected Charge...", "expected_type": "Problem", "predicted_type": "Problem", "category_correct": true, "ml": {"answer": "**Answer:** \\nTo assist with the unexpected charge on your latest billing statement, please provide your account number, the date of the statement, and a detailed description of the discrepancy. This information will help us investigate and resolve the issue promptly.\\n\\n**Steps:**\\n\\n1. Gather the requi", "retrieval": {"type_precision": 1.0, "type_diversity": 1}, "similarity": 0.767}, "no_ml": {"answer": "**Answer:** We apologize for the unexpected charge on your latest invoice. To investigate this matter further, could you please provide us with details about the charge, including the date, amount, and any relevant information?\\n\\n**Steps:**\\n\\n1. Please provide your account number and a detailed descri", "retrieval": {"type_precision": 0.6, "type_diversity": 2}, "similarity": 0.926}}
]'''
FALLBACK_RESULTS = json.loads(FALLBACK_RESULTS_JSON)

In [4]:
def retrieval_metrics(docs, expected_type):
    counts = {}
    for d in docs:
        t = d.metadata.get('type', 'unknown')
        counts[t] = counts.get(t, 0) + 1
    total = len(docs)
    expected = counts.get(expected_type, 0)
    return {'type_precision': expected / total if total else 0, 'type_diversity': len(counts)}

async def emb_sim(text1, text2):
    embed = settings.embedding_model
    v1, v2 = await asyncio.gather(embed.aembed_query(text1), embed.aembed_query(text2))
    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))

async def run_pipeline(query, category=None):
    docs = await Retriever().retrieve_docs(query, category)
    answer = await Generator().generate(query, category)
    return docs, answer

def display_report(results):
    valid = [r for r in results if 'error' not in r]
    if not valid:
        print('No valid results.'); return

    ca = np.mean([r['category_correct'] for r in valid])
    mt = np.mean([r['ml']['retrieval']['type_precision'] for r in valid])
    nt = np.mean([r['no_ml']['retrieval']['type_precision'] for r in valid])
    md = np.mean([r['ml']['retrieval']['type_diversity'] for r in valid])
    nd = np.mean([r['no_ml']['retrieval']['type_diversity'] for r in valid])
    ms = np.mean([r['ml']['similarity'] for r in valid])
    ns = np.mean([r['no_ml']['similarity'] for r in valid])

    print('=' * 72)
    print('  RAG EVALUATION: WITH SVM CLASSIFIER  vs  WITHOUT CLASSIFIER')
    print('=' * 72)
    print(f'  Valid queries: {len(valid)}/{len(results)}  |  SVM accuracy: {ca:.0%}')
    print()
    print(f'  {"Metric":<32} {"With ML":>8} {"Without ML":>10}')
    print(f'  {"-"*32} {"-"*8} {"-"*10}')
    print(f'  {"Retrieval: Type Precision":<32} {mt:>7.0%}  {nt:>9.0%}')
    print(f'  {"Retrieval: Type Diversity":<32} {md:>8.1f} {nd:>10.1f}')
    print(f'  {"Embedding Similarity (vs GT)":<32} {ms:>8.3f} {ns:>10.3f}')
    print()
    wm = sum(1 for r in valid if r['ml']['similarity'] > r['no_ml']['similarity'] + 0.02)
    wn = sum(1 for r in valid if r['no_ml']['similarity'] > r['ml']['similarity'] + 0.02)
    print(f'  Generation: ML wins {wm}, NoML wins {wn}, Ties {len(valid)-wm-wn}')
    print('=' * 72)
    print()
    print('  Per-query:')
    for r in valid:
        t = 'OK ' if r['category_correct'] else 'MIS'
        ic = '+' if r['ml']['similarity'] > r['no_ml']['similarity'] else '-'
        print(f'  [{t}] {r["expected_type"]:>8}->{r["predicted_type"]:>8}  '
              f'ML_tp={r["ml"]["retrieval"]["type_precision"]:.0%} '
              f'NoML_tp={r["no_ml"]["retrieval"]["type_precision"]:.0%}  '
              f'ML_sim={r["ml"]["similarity"]:.3f} NoML_sim={r["no_ml"]["similarity"]:.3f}  {ic}')
    print()

    print('=' * 72)
    print('  SIDE-BY-SIDE ANSWERS')
    print('=' * 72)
    for r in valid:
        print(f'\nQ: {r["query"][:80]}...')
        print(f'  Type={r["expected_type"]} SVM->{r["predicted_type"]}')
        print(f'  [GT]    {r["no_ml"]["answer"][:150]}...')
        print(f'  [With]  {r["ml"]["answer"][:150]}...')
        print(f'  [NoML]  {r["no_ml"]["answer"][:150]}...')

In [6]:
async def main():
    seed(42)
    df = pd.read_csv(BACKEND_DIR / 'data' / 'clean.csv')
    test_set = []
    for t in TYPES:
        subset = df[df['type'] == t].dropna(subset=['text', 'answer']).sample(n=2, random_state=hash(t)%1000)
        for _, row in subset.iterrows():
            test_set.append({'question': str(row['text']).strip(), 'ground_truth': str(row['answer']).strip(), 'type': str(row['type']).strip()})
    shuffle(test_set)

    clf = Classifier()
    results = []
    hit_rate_limit = False

    for i, entry in enumerate(test_set):
        if hit_rate_limit:
            results.append({'query': entry['question'][:100], 'expected_type': entry['type'], 'error': 'skipped (prev rate limit)'})
            continue
        print(f'[{i+1}/8] {entry["type"]}: {entry["question"][:60]}...')
        try:
            pred = await clf.classify(entry['question'])
            ml_d, ml_a = await run_pipeline(entry['question'], pred)
            no_d, no_a = await run_pipeline(entry['question'], None)
            mr = retrieval_metrics(ml_d, entry['type'])
            nr = retrieval_metrics(no_d, entry['type'])
            ml_s = await emb_sim(ml_a, entry['ground_truth'])
            no_s = await emb_sim(no_a, entry['ground_truth'])
            results.append({'query': entry['question'], 'expected_type': entry['type'], 'predicted_type': pred,
                'category_correct': pred == entry['type'],
                'ml': {'answer': ml_a, 'retrieval': mr, 'similarity': ml_s},
                'no_ml': {'answer': no_a, 'retrieval': nr, 'similarity': no_s}})
            print(f'    -> {pred}, sim: ML={ml_s:.3f} NoML={no_s:.3f}')
        except Exception as e:
            msg = str(e)
            if '429' in msg or 'rate_limit' in msg.lower():
                hit_rate_limit = True
            print(f'    ERROR: {msg[:100]}')
            results.append({'query': entry['question'][:200], 'expected_type': entry['type'], 'error': msg[:200]})
        time.sleep(8)

    return results

fresh_results = await main()
fresh_valid = [r for r in fresh_results if 'error' not in r]

if len(fresh_valid) >= 4:
    print('\n*** Fresh run complete. Using live results. ***\n')
    results = fresh_results
else:
    print(f'\n*** Only {len(fresh_valid)}/8 queries succeeded (rate limited). Using stored results from last run. ***\n')
    results = FALLBACK_RESULTS

display_report(results)

[1/8] Request: Concerning Medical Data Security in a Hospital Environment U...
loading knowledge...
loading knowledge...
loading knowledge...
loading knowledge...
    -> Request, sim: ML=0.867 NoML=0.804
[2/8] Problem: Problem with Investment Analytics Dashboard
dashboard showin...
loading knowledge...
loading knowledge...
loading knowledge...
loading knowledge...
    -> Problem, sim: ML=0.758 NoML=0.768
[3/8] Change: Support Request for Integrating DocuSign with JIRA Software
...
loading knowledge...
loading knowledge...
loading knowledge...
loading knowledge...
    -> Change, sim: ML=0.950 NoML=0.915
[4/8] Change: Respectful support, please highlight the importance of enhan...
loading knowledge...
loading knowledge...
loading knowledge...
loading knowledge...
    -> Change, sim: ML=0.758 NoML=0.762
[5/8] Request: Customer Support, I am requesting detailed integration instr...
loading knowledge...
loading knowledge...
loading knowledge...
loading knowledge...
    -> Request, sim: ML=0